## FAISS

Facebook AI Similarity Search(FAISS) is library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM . It also contains supporting code for evaluation and parameter tuning

In [ ]:
from langchain_community.document_loaders import TextLoader,PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter



# loader = TextLoader("speech.txt")
loader = PyPDFLoader("Abhishek Jaiswar - Web Developer.pdf")
documents = loader.load()
# text_splitter = CharacterTextSplitter(chunk_size=500,chunk_overlap=30)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
docs = text_splitter.split_documents(documents)


In [ ]:
docs

In [ ]:
embeddings =  OllamaEmbeddings(
    model="qwen3-embedding:0.6b"
)
# Save to FAISS vector store
# db = FAISS.from_documents(docs,embeddings)
# db

# Save locally

db = FAISS.from_documents(docs, embeddings)
db.save_local("faiss_index")

# Load from local directory
db = FAISS.load_local("faiss_index", embeddings,allow_dangerous_deserialization=True)
db


In [ ]:
## querying

query = "Abhishek Skills ?"
docs = db.similarity_search(query)
docs[0].page_content

In [ ]:
query="Abhishek Projects"
docs = db.similarity_search(query)
docs[0].page_content

In [ ]:
## Retriever
# We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers


In [ ]:
retriver = db.as_retriever()
retriver.invoke(query)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama

# 5. LLM + prompt for generation
llm = Ollama(model="gemma3:1b")

prompt = ChatPromptTemplate.from_template(
    """Answer the question based ONLY on the following context:
{context}

Question: {question}"""
)
chain = (
    {"context": retriver, "question": lambda x: x}  # RunnablePassthrough for question
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Query
response = chain.invoke("Who is Abhishek Jaiswar ?")
print(response)

In [ ]:
response = chain.invoke("what do you think about Abhishek Jaiswar ?")
print(response)